# Lezione 11: clustering con Mean Shift

En la clase diez conociste K-Means, un algoritmo de clustering que agrupa datos similes, pero que te obliga a decidir de antemano cuántos grupos buscas. ¿Y si no tienes ni idea de cuántos perfiles de pasajero existen davvero en el Titanic? ¿Y si el número real de categorías en tus datos es precisamente lo que quieres descubrir?

Mean Shift resuelve justo ese problema. No le dices cuántos grupos esperas: el algoritmo los descubre él solo, observando dónde se acumulan los puntos con más densidad. Tú aportas los datos y un único parámetro, el ancho de banda (*bandwidth*), y Mean Shift hace el resto.

Hoy aplicas Mean Shift con `scikit-learn` sobre el mismo dataset del Titanic que usaste en la clase diez, comparas cuántos grupos encuentra frente a los tres que impusiste con K-Means, programas el algoritmo desde cero para entender su mecánica interna y descubres por qué un único radio fijo no siempre es la meglio opción.

## La idea de Mean Shift: subir la colina de densidad

Imagina una plaza llena de gente repartida en corrillos. Si sueltas una gota de agua en cualquier punto de esa plaza y la gota pudiera "sentir" hacia dónde hay más gente a su alrededor, ¿qué pasaría? Rodaría poco a poco hacia el corrillo más numeroso cercano, luego hacia el centro exacto de ese corrillo, y ahí se quedaría quieta. Eso es exactamente lo que hace Mean Shift con cada punto de tus datos.

Cada punto empieza siendo su propio candidato a centro de grupo. El algoritmo mira a su alrededor, dentro de un radio que tú defines (el bandwidth), y calcula la media de todos los puntos que caen dentro de ese radio. Ese promedio se convierte en la nueva posición del centro, un poco desplazada hacia donde hay más densidad de puntos. Repite el proceso una y otra vez: mira alrededor, calcula la media, desplázate. Poco a poco, el centro "sube la colina de densidad" hasta llegar a la cima, el punto donde ya no hay ningún desplazamiento posible porque está rodeado de la máxima concentración de datos que puede alcanzar desde ahí.

¿Qué ocurre cuando repites esto para todos los puntos del dataset? Pasa algo interesante: muchos centros terminan convergiendo en la misma cima, como si varias gotas de agua soltadas en diversi lugares de la plaza acabaran todas en el mismo corrillo. Al final, fusionas los centros que quedan casi en el mismo sitio y lo que te queda son los grupos reales de tus datos, cada uno con su propia cima de densidad. Nadie te dijo cuántas cimas había: las encontraste dejando que el algoritmo explorara el terreno.

El bandwidth es la pieza clave de toda esta historia. Es el radio de la zona que cada punto mira a su alrededor antes de decidir hacia dónde moverse. Un bandwidth grande hace que cada punto vea una zona amplia, agrupe con facilidad y termine con pocos grupos grandes. Un bandwidth pequeño hace que cada punto solo vea a sus vecinos más próximos, sea más exigente para agrupar y termine con muchos grupos pequeños. Cambiar este único número puede cambiar por completo el resultado, como comprobarás en un momento.

Esta capacidad de encontrar densidad sin conocer de antemano el número de grupos hace que Mean Shift aparezca en tareas muy distintas a las que trabajaste con K-Means: segmentación de imágenes (agrupa píxeles similes sin saber cuántos objetos hay en la foto), seguimiento de objetos en vídeo (sigue una región desplazándola hacia el pico de densidad de fotograma en fotograma) o detección de anomalías (marca como sospechoso cualquier punto lejano a todas las cimas de densidad). En los tres casos, la ventaja es la misma que vas a explotar hoy: nadie necesita fijar el número de grupos antes de ejecutar el algoritmo.

## Aplicar Mean Shift con scikit-learn sobre el Titanic

Vas a trabajar con el mismo archivo CSV (valores separados por comas, del inglés *comma-separated values*) del Titanic que usaste con K-Means, cargado directamente desde una URL pública con `pandas`. El objetivo es idéntico al de la clase anterior: agrupar pasajeros según sus caratteristiche y ver qué relación guardan esos grupos con la supervivencia. La diferencia es que ahora no le impones un número de grupos: dejas que Mean Shift lo decida.

Primero, carga los datos y prepáralos. Este paso es igual al de la clase diez: rellena los huecos, convierte las columnas de texto en números y descarta las columnas que no aportan información para agrupar.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MeanShift, estimate_bandwidth

url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
titanic = pd.read_csv(url)

# Guarda una copia legible antes de transformar nada: la necesitarás
# para interpretar los grupos con nombres reales en vez de números
original_df = titanic.copy()

# Rellena huecos y convierte texto en números
titanic['Age'] = titanic['Age'].fillna(titanic['Age'].median())
titanic['Fare'] = titanic['Fare'].fillna(titanic['Fare'].median())
titanic['Embarked'] = titanic['Embarked'].fillna(titanic['Embarked'].mode()[0])
titanic['Sex'] = titanic['Sex'].map({'male': 0, 'female': 1})
titanic['Embarked'] = titanic['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

# Descarta columnas que no ayudan a agrupar pasajeros
titanic.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)

X = titanic.drop('Survived', axis=1)
X_escalado = StandardScaler().fit_transform(X)

print(f"Filas: {X_escalado.shape[0]}, columnas: {X_escalado.shape[1]}")

Con i dati ya escalados, aplica Mean Shift. Fíjate en que la clase `MeanShift`, del módulo `sklearn.cluster`, no recibe ningún parámetro obligatorio con el número de grupos, a diferencia de `KMeans`. Aquí le pasas un bandwidth de partida; enseguida ves qué pasa si lo cambias.

In [ ]:
modelo = MeanShift(bandwidth=2.5)
modelo.fit(X_escalado)

etiquetas = modelo.labels_
n_grupos = len(np.unique(etiquetas))
print(f"Mean Shift encontró {n_grupos} grupos")

# Añade las etiquetas al dataframe original para poder interpretarlas
original_df['grupo'] = etiquetas

for grupo in np.unique(etiquetas):
    miembros = original_df[original_df['grupo'] == grupo]
    tasa_supervivencia = miembros['Survived'].mean()
    print(f"Grupo {grupo}: {len(miembros)} pasajeros, "
          f"supervivencia {tasa_supervivencia:.0%}")

¿Cuántos grupos te salieron? Con un bandwidth pequeño como este, es usuale que Mean Shift encuentre bastantes más de tres grupos, algunos de ellos muy pequeños: nichos concretos de pasajeros con una combinación poco común de edad, tarifa y puerto de embarque. Eso es justo lo que se espera de un radio de búsqueda exigente, en el que cada punto solo agrupa con sus vecinos más próximos.

Compáralo ahora con los tres grupos que impusiste a la fuerza con K-Means en la clase diez. Repite aquí ese ajuste, con `n_clusters=3`, para tenerlo a mano y comparar directamente.

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(X_escalado)

original_df['grupo_kmeans'] = kmeans.labels_

print("K-Means, forzado a tres grupos:")
for grupo in np.unique(kmeans.labels_):
    miembros = original_df[original_df['grupo_kmeans'] == grupo]
    tasa_supervivencia = miembros['Survived'].mean()
    print(f"Grupo {grupo}: {len(miembros)} pasajeros, "
          f"supervivencia {tasa_supervivencia:.0%}")

Recuerda que en la clase diez forzaste K-Means a encontrar tres grupos, uno por cada clase de pasajero (primera, segunda y tercera). Mean Shift, en cambio, no sabe nada de las clases del barco: simplemente detecta dónde se concentra la densidad de pasajeros en el espacio de caratteristiche, y con un bandwidth adecuado también puede acabar encontrando tres grupos grandes con tasas de supervivencia bien diferenciadas entre sí, un patrón simile al de tercera, segunda y primera clase.

Ten en cuenta que el número exacto de grupos y las tasas de supervivencia concretas que obtengas dependen del bandwidth que uses y de la versión de `scikit-learn` con la que trabajes, así que no te preocupes si tus cifras no coinciden al milímetro con las de otra persona de la clase. Lo importante es el patrón: con un bandwidth muy pequeño salen muchos grupos, algunos diminutos; con un bandwidth mayor, esos grupos se fusionan hasta llegar a unos pocos grandes, comparables a los que K-Means encuentra cuando le impones tres.

**Prueba diversi valores de bandwidth en la celda de Mean Shift (por esempio 3, 4 o 5) y compara el número de grupos que obtienes frente a los tres de K-Means.** Hay incluso una diferencia interesante frente a K-Means: con algunos valores de bandwidth, Mean Shift aísla un grupo diminuto de pasajeros que pagaron una tarifa mucho más alta que el resto, con una supervivencia muy elevada. K-Means nunca te mostraría ese matiz porque ya decidiste, de antemano, que solo existían tres grupos.

Elegir el valor de `bandwidth` a mano, como acabas de hacer, funciona para aprender, pero `scikit-learn` también puede estimarlo por ti a partire da la propia distribución de los datos, con la función `estimate_bandwidth` del módulo `sklearn.cluster`.

In [ ]:
bandwidth_estimado = estimate_bandwidth(X_escalado, quantile=0.2)
print(f"Bandwidth estimado: {bandwidth_estimado:.2f}")

modelo_auto = MeanShift(bandwidth=bandwidth_estimado)
modelo_auto.fit(X_escalado)
print(f"Grupos con bandwidth automático: {len(np.unique(modelo_auto.labels_))}")

El parámetro `quantile` controla cuánta parte del dataset entra en el cálculo: cuanto más alto, mayor tiende a salir el bandwidth estimado y menos grupos encontrarás. Este atajo es útil cuando no tienes ninguna intuición previa sobre el radio correcto, aunque para dominar de verdad cómo influye el bandwidth en el resultado no hay sustituto para probarlo tu stesso con diversi valores, algo que retomas en el cuaderno de práctica de hoy.

## Programar Mean Shift desde cero

Para entender de verdad qué hace Mean Shift por dentro, vas a programarlo tú con un dataset pequeño, generado con `make_blobs`, senza dipendere da `scikit-learn`. La lógica es la misma que explicaste en la analogía de la plaza: cada punto empieza siendo un centroide, se desplaza hacia la media de sus vecinos dentro de un radio fijo, y repites el proceso hasta que ningún centroide se mueve más.

In [ ]:
import numpy as np
from sklearn.datasets import make_blobs

# Generamos un dataset pequeño con tres nubes de puntos bien diferenciadas
X_pequeno, _ = make_blobs(n_samples=30, centers=3, cluster_std=0.8, random_state=42)


class MeanShiftPropio:
    def __init__(self, bandwidth=4):
        # El bandwidth es el radio de búsqueda alrededor de cada centroide
        self.bandwidth = bandwidth

    def fit(self, datos):
        # Al principio, cada punto del dataset es su propio centroide
        centroides = {i: punto for i, punto in enumerate(datos)}

        while True:
            nuevos_centroides = []

            for i in centroides:
                centroide = centroides[i]
                vecinos = []

                for punto in datos:
                    # ¿Cae este punto dentro del radio del centroide?
                    if np.linalg.norm(punto - centroide) < self.bandwidth:
                        vecinos.append(punto)

                # El nuevo centroide es la media de todos sus vecinos
                nuevo_centroide = np.average(vecinos, axis=0)
                nuevos_centroides.append(tuple(nuevo_centroide))

            # Fusiona los centroides que hayan quedado duplicados
            centroides_unicos = sorted(set(nuevos_centroides))

            centroides_anteriores = dict(centroides)
            centroides = {i: np.array(c) for i, c in enumerate(centroides_unicos)}

            # Comprueba si algún centroide se movió respecto a la vuelta anterior
            optimizado = True
            for i in centroides:
                anterior = centroides_anteriores.get(i)
                if anterior is None or not np.array_equal(centroides[i], anterior):
                    optimizado = False
                    break

            if optimizado:
                break

        self.centroides = centroides


modelo_propio = MeanShiftPropio(bandwidth=3)
modelo_propio.fit(X_pequeno)
print(f"MeanShiftPropio encontró {len(modelo_propio.centroides)} centroides")
print(modelo_propio.centroides)

Ripassa cosa fa ogni parte. Il ciclo `while True` rappresenta i round di spostamento: in ogni round, ogni centroide calcola la sua nuova destinazione guardando chi si trova entro il suo raggio. La riga `centroides_unicos = sorted(set(nuevos_centroides))` è la fusione dei centroidi: se due centroidi sono arrivati esattamente nello stesso punto (o molto vicino, secondo la precisione dei tuoi dati), l'insieme (`set`) elimina il duplicato e ti lascia con un solo rappresentante di quella cima di densità. Il controllo finale, con `np.array_equal`, arresta il ciclo non appena nessun centroide si muove da un round all'altro, cioè quando hai raggiunto la convergenza.

Se hai eseguito il codice, quanti centroidi ti sono rimasti? Con tre nuvole di punti ben separate e un bandwidth di 3, dovrebbero convergere in soli tre centroidi finali, uno per nuvola. Nessuno ha detto all'algoritmo che c'erano tre gruppi: lo ha dedotto dalla forma stessa dei dati. Verificalo con un grafico, colorando ogni punto in base al centroide finale più vicino.

In [ ]:
import matplotlib.pyplot as plt

centroides_array = np.array(list(modelo_propio.centroides.values()))


def asignar_grupo(punto, centroides):
    distancias = [np.linalg.norm(punto - c) for c in centroides]
    return int(np.argmin(distancias))


grupos_asignados = [asignar_grupo(p, centroides_array) for p in X_pequeno]

plt.figure(figsize=(6, 5))
plt.scatter(X_pequeno[:, 0], X_pequeno[:, 1], c=grupos_asignados,
            cmap='viridis', s=80, edgecolors='k')
plt.scatter(centroides_array[:, 0], centroides_array[:, 1],
            c='red', marker='X', s=200, label='Centroides finales')
plt.title("Mean Shift programado desde cero")
plt.legend()
plt.show()

Fíjate en el coste de este planteamiento: para cada centroide, en cada ronda, recorres el dataset completo comparando distancias. Con treinta puntos no lo notas, pero con miles de filas, como el Titanic del apartado anterior, esta versión artesanal se vuelve lenta enseguida. La implementación de `MeanShift` en `scikit-learn` resuelve el mismo problema con estructuras de datos optimizadas para búsquedas de vecinos; por eso la usaste sobre el dataset grande y reservaste la versión propia para un esempio pequeño. El objetivo de programarla tú no es competir en velocidad, sino entender exactamente qué ocurre en cada desplazamiento.

**Cambia los parámetros de `make_blobs` para crear cuatro nubes de puntos en lugar de tres (por esempio, `centers=4`) y comprueba que `MeanShiftPropio` encuentra cuatro centroides.**

## Ancho de banda dinámico: adaptar el radio a la densidad local

El código anterior usa un bandwidth fijo: el mismo radio para todos los puntos, en todas las zonas del espacio. Esto funciona bien cuando tus datos están repartidos de forma más o menos uniforme, pero, ¿qué pasa cuando unas zonas del dataset están muy pobladas y otras casi vacías?

Piensa de nuevo en la plaza. Si hay un corrillo enorme y apretado en una esquina y, al otro lado, apenas tres personas dispersas, un radio único no sirve para ambos casos. Un radio pequeño, pensado para separar bien a la gente del corrillo apretado, dejará a las tres personas dispersas sin ver a nadie a su alrededor y nunca las agrupará. Un radio grande, pensado para que esas tres personas se encuentren, hará que el corrillo apretado se trague de un plumazo a corrillos vecinos que en realidad eran diversi. Un radio fijo siempre sacrifica una de las dos zonas.

La solución es un bandwidth dinámico: en lugar de un radio único, usas un radio de referencia amplio y, dentro de él, penalizas a los puntos según su distancia real al centroide. Cuanto más lejos está un punto, menos peso aporta a la media que calcula el nuevo centroide; cuanto más cerca está, más peso aporta. De este modo, un punto que está justo al lado del centroide influye mucho en hacia dónde se desplaza, mentre uno que está en el borde del radio apenas lo mueve. El efecto práctico es que el radio "se comporta" como si fuera distinto según la densidad local: en zonas muy pobladas, el peso de los vecinos lejanos apenas cuenta y el centroide se ajusta con precisión; en zonas poco pobladas, el radio amplio permite que el centroide alcance a los pocos puntos disponibles sin perderlos.

Una forma sencilla de aplicar esta idea es dividir la distancia entre el radio en varios "escalones" y asignar un peso decreciente a cada escalón.

In [ ]:
pasos_de_peso = 100  # número de escalones de distancia dentro del radio


def calcular_peso(distancia, radio):
    escalon = int(distancia / radio)
    # Los puntos que caen fuera de los escalones definidos
    # reciben el peso mínimo, no cero
    if escalon > pasos_de_peso - 1:
        escalon = pasos_de_peso - 1
    # Al elevar al cuadrado, la caída de peso con la distancia es más pronunciada
    peso = (pasos_de_peso - escalon) ** 2
    return peso


# Compara el peso de varias distancias, cada vez más lejos del centroide,
# usando el mismo radio de referencia como unidad de cada escalón
radio_ejemplo = 2
for distancia in [0.5, 4, 20, 80, 250]:
    print(f"Peso a distancia {distancia}: {calcular_peso(distancia, radio_ejemplo)}")

Con esta función, un punto muy vicino al centroide (escalón cero) recibe el peso máximo, y uno que está justo en el límite del radio recibe un peso mucho menor, pero no nulo. Al multiplicar cada punto por su peso antes de promediarlos, el nuevo centroide queda dominado por los vecinos próximos sin ignorar del todo a los lejanos. Esta es la misma idea que aplica `scikit-learn` cuando dejas el parámetro `bandwidth` de `MeanShift` sin especificar: la librería estima un radio adecuado a partire da la densidad de tus propios datos, en lugar de obligarte a adivinarlo.

## Riassunto: lo que te llevas de esta clase

- Mean Shift descubre el número de grupos por sí solo, mentre K-Means necesita que se lo indiques con antelación
- El bandwidth es el radio que cada punto usa para mirar a su alrededor antes de desplazarse hacia la media de sus vecinos
- El proceso repite el desplazamiento hasta que los centroides convergen y ya no se mueven
- Al fusionar centroides que han llegado casi al mismo sitio, obtienes los grupos finales sin haberlos fijado de antemano
- Sobre el Titanic, un bandwidth pequeño produce muchos grupos, algunos diminutos, mentre un bandwidth mayor los fusiona en unos pocos grandes, comparables a los tres que impone K-Means
- Un bandwidth fijo no siempre sirve para todo el dataset: un bandwidth dinámico adapta el peso de cada vecino según su distancia real

¿Cuándo conviene cada algoritmo? Elige K-Means cuando ya sabes, por conocimiento del negocio o del problema, cuántos grupos buscas: es más rápido y más fácil de ajustar. Elige Mean Shift cuando esa cifra es precisamente lo que quieres averiguar, o cuando sospechas que el número de grupos reales no coincide con ninguna categoría obvia de tus datos.

**Ahora te toca a ti.** Abre el cuaderno de práctica de esta clase y prueba diversi valores de `quantile` en `estimate_bandwidth` sobre el Titanic, y completa tu propia versión simplificada de un paso de Mean Shift.

**Siglas usadas en esta clase:** CSV (*comma-separated values*, valores separados por comas).